# ASL Encoder-Decoder Transformer Training (Kaggle)

Trains a Transformer encoder-decoder for continuous sign language recognition
on the **How2Sign MediaPipe+CLIP** dataset.

## Persistence across sessions

Checkpoints are saved to `/kaggle/working/checkpoints/` during training.
To **resume** a previous run in a new session:

1. After training finishes (or is interrupted), run the **"Upload Checkpoints"** cell at the bottom.
   It creates/updates a private Kaggle dataset called `asl-encdec-checkpoints`.
2. In the next session, add that dataset as an input — its files appear at
   `/kaggle/input/asl-encdec-checkpoints/`.
3. The resume logic below will pick up the latest checkpoint automatically.

> **Google Drive alternative**: see the optional cells at the bottom.

## 1 — Configuration

In [ ]:
# ───────────────────────── User config ─────────────────────────
KAGGLE_USERNAME       = "YOUR_KAGGLE_USERNAME"   # <-- set this!
CHECKPOINT_DATASET    = "asl-encdec-checkpoints"  # private dataset for checkpoints

# Paths
WORKING_CKPT_DIR      = "/kaggle/working/checkpoints"
INPUT_CKPT_DIR        = f"/kaggle/input/{CHECKPOINT_DATASET}"

# Auto-discover the HDF5 file under /kaggle/input/
import os, glob as _glob
_H5_FILENAME = "how2sign_mediapipe_clip_200vocab.h5"
_candidates = _glob.glob(f"/kaggle/input/**/{_H5_FILENAME}", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
    print(f"Found dataset: {DATA_PATH}")
else:
    # Fallback — edit this if auto-discovery fails
    DATA_PATH = f"/kaggle/input/asl-mediapipe-clip-dataset/{_H5_FILENAME}"
    print(f"WARNING: auto-discovery found nothing. Using fallback: {DATA_PATH}")
    # Debug: show what's actually mounted
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print(f"  {os.path.join(root, f)}")
        if root.count(os.sep) > 4:
            break

# Hyperparameters — tuned for 16 GB datacenter GPU (T4 / P100)
HIDDEN_DIM            = 512
NUM_ENCODER_LAYERS    = 6
NUM_DECODER_LAYERS    = 6
NUM_HEADS             = 8
DROPOUT               = 0.1
USE_CHANNEL_ATTENTION = True
ATTENTION_REDUCTION   = 8

BATCH_SIZE            = 32
EVAL_BATCH_SIZE       = 32
EPOCHS                = 100
LR                    = 3e-4
WARMUP_EPOCHS         = 5
GRAD_ACCUM_STEPS      = 2      # effective batch = 32 × 2 = 64
MAX_SEQ_LEN           = 512    # cap input frames — prevents quadratic attention OOM
NUM_WORKERS           = 0      # data is preloaded into RAM — no I/O workers needed
H5_CACHE_MB           = 256
USE_AMP               = True   # mixed precision — essential for fitting large batches
RESUME                = True   # auto-resume from latest checkpoint

# Checkpoint frequency: save every N epochs (1 = every epoch, for crash safety)
CKPT_EVERY_N_EPOCHS   = 1

## 2 — Install dependencies & imports

In [ ]:
!pip install -q h5py tqdm

In [ ]:
import os, json, math, shutil, glob
from functools import partial

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm.auto import tqdm

print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3 — Checkpoint persistence helpers

In [ ]:
os.makedirs(WORKING_CKPT_DIR, exist_ok=True)


def _find_latest_checkpoint():
    """Return the path to the most recent `latest_checkpoint.pt`, or None."""
    candidates = [
        os.path.join(WORKING_CKPT_DIR, "latest_checkpoint.pt"),
        os.path.join(INPUT_CKPT_DIR,   "latest_checkpoint.pt"),
    ]
    for path in candidates:
        if os.path.isfile(path):
            print(f"Found checkpoint: {path}")
            return path
    print("No existing checkpoint found — training from scratch.")
    return None


def _copy_input_checkpoints_to_working():
    """Copy checkpoint files from the read-only input dataset to the writable working dir."""
    if not os.path.isdir(INPUT_CKPT_DIR):
        return
    for fname in os.listdir(INPUT_CKPT_DIR):
        if fname.endswith(".pt"):
            src = os.path.join(INPUT_CKPT_DIR, fname)
            dst = os.path.join(WORKING_CKPT_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"  Copied {fname} → working dir")


# Eagerly copy so that saves (best_model, periodic) can overwrite in-place.
_copy_input_checkpoints_to_working()
RESUME_PATH = _find_latest_checkpoint() if RESUME else None

## 4 — Model: Encoder-Decoder Transformer

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding with dynamic expansion."""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        pe = self._compute_pe(max_len, d_model)
        self.register_buffer('pe', pe)

    def _compute_pe(self, length, d_model):
        pe = torch.zeros(length, d_model)
        position = torch.arange(0, length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len > self.pe.size(0):
            new_max_len = max(seq_len, self.pe.size(0) * 2)
            new_pe = self._compute_pe(new_max_len, self.d_model).to(self.pe.device)
            self.register_buffer('pe', new_pe)
        return x + self.pe[:seq_len, :].unsqueeze(0)


class ChannelAttention(nn.Module):
    """Lightweight channel attention over feature dimensions."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x):
        pooled = x.mean(dim=1)
        weights = torch.sigmoid(self.fc2(F.relu(self.fc1(pooled))))
        return x * weights.unsqueeze(1)


class SignLanguageTransformer(nn.Module):
    """
    Encoder-Decoder Transformer for continuous sign language recognition.
    Video features → Encoder → Decoder → gloss sequence (autoregressive).
    """
    def __init__(self, num_classes, input_features=512, hidden_dim=256,
                 num_encoder_layers=4, num_decoder_layers=4, num_heads=8,
                 dropout=0.1, max_glosses=100, use_channel_attention=False,
                 attention_reduction=8):
        super().__init__()
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.max_glosses = max_glosses
        self.use_channel_attention = use_channel_attention
        self.pad_idx = 0
        self.sos_idx = 1
        self.eos_idx = 2

        self.channel_attention = (
            ChannelAttention(input_features, reduction=attention_reduction)
            if use_channel_attention else None
        )
        self.input_proj = nn.Sequential(
            nn.Linear(input_features, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim),
        )
        self.pos_encoder = PositionalEncoding(hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads, dim_feedforward=hidden_dim * 4,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)

        self.gloss_embedding = nn.Embedding(num_classes + 3, hidden_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim, nhead=num_heads, dim_feedforward=hidden_dim * 4,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_decoder_layers)

        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes + 3),
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, video_features):
        if self.channel_attention is not None:
            video_features = self.channel_attention(video_features)
        x = self.input_proj(video_features)
        x = self.pos_encoder(x)
        return self.encoder(x)

    def decode_step(self, memory, tgt_input, tgt_mask=None):
        tgt = self.gloss_embedding(tgt_input)
        tgt = self.pos_encoder(tgt)
        output = self.decoder(tgt, memory, tgt_mask=tgt_mask)
        return self.output_proj(output)

    def forward(self, video_features, gloss_targets=None):
        memory = self.encode(video_features)
        if gloss_targets is not None:
            tgt_len = gloss_targets.size(1)
            tgt_mask = self.generate_square_subsequent_mask(tgt_len).to(video_features.device)
            return self.decode_step(memory, gloss_targets, tgt_mask)
        return self.generate(memory)

    def generate(self, memory, max_length=None):
        if max_length is None:
            max_length = self.max_glosses
        batch_size = memory.size(0)
        device = memory.device
        generated = torch.full((batch_size, 1), self.sos_idx, dtype=torch.long, device=device)
        for _ in range(max_length):
            tgt_len = generated.size(1)
            tgt_mask = self.generate_square_subsequent_mask(tgt_len).to(device)
            logits = self.decode_step(memory, generated, tgt_mask)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == self.eos_idx).all():
                break
        return generated[:, 1:]

    @staticmethod
    def generate_square_subsequent_mask(sz):
        mask = torch.triu(torch.ones(sz, sz), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))


def prepare_targets(labels, label_lengths, sos_idx=1, eos_idx=2, pad_idx=0):
    batch_size = labels.size(0)
    device = labels.device
    max_len = label_lengths.max().item()
    adjusted_labels = labels.clone()
    for i in range(batch_size):
        L = label_lengths[i].item()
        adjusted_labels[i, :L] = labels[i, :L] + 3
    decoder_input = torch.full((batch_size, max_len + 1), pad_idx, dtype=torch.long, device=device)
    decoder_input[:, 0] = sos_idx
    for i in range(batch_size):
        L = label_lengths[i].item()
        decoder_input[i, 1:L+1] = adjusted_labels[i, :L]
    decoder_target = torch.full((batch_size, max_len + 1), pad_idx, dtype=torch.long, device=device)
    for i in range(batch_size):
        L = label_lengths[i].item()
        decoder_target[i, :L] = adjusted_labels[i, :L]
        decoder_target[i, L] = eos_idx
    target_lengths = label_lengths + 1
    return decoder_input, decoder_target, target_lengths


print("Model classes defined.")

## 5 — Dataset & DataLoader

In [ ]:
class H5Dataset(Dataset):
    """HDF5 dataset — preloads all sequences into RAM for fast GPU feeding."""

    def __init__(self, h5_path, split='train', max_seq_len=None, h5_cache_mb=256):
        self.split = split
        self.max_seq_len = max_seq_len

        print(f"[{split}] Loading dataset into RAM...", flush=True)
        with h5py.File(h5_path, 'r') as f:
            total_raw = len(f[f'{split}_sequences'])

            if 'num_classes' in f.attrs:
                self.num_classes = int(f.attrs['num_classes'])
            elif 'num_classes' in f:
                self.num_classes = int(f['num_classes'][()])
            else:
                raise ValueError("Cannot find num_classes in HDF5 file")

            sample_seq = f[f'{split}_sequences'][0]
            self.feature_dim = sample_seq.shape[1] if len(sample_seq.shape) == 2 else int(f.attrs.get('feature_dim', 512))

            if 'gloss_names' in f:
                gloss_names = f['gloss_names'][:]
                self.idx_to_gloss = {
                    i: name.decode('utf-8') if isinstance(name, bytes) else name
                    for i, name in enumerate(gloss_names)
                }
            elif 'gloss_to_idx' in f.attrs:
                gloss_mapping = json.loads(f.attrs['gloss_to_idx'])
                self.idx_to_gloss = {v: k for k, v in gloss_mapping.items()}
            elif 'gloss_to_idx' in f:
                gloss_mapping = json.loads(f['gloss_to_idx'][()])
                self.idx_to_gloss = {v: k for k, v in gloss_mapping.items()}
            else:
                self.idx_to_gloss = {i: f'class_{i}' for i in range(self.num_classes)}

            seq_lengths = f[f'{split}_sequence_lengths'][:].astype(np.int32)
            max_seq_width = int(f[f'{split}_sequences'].shape[1])
            seq_lengths = np.clip(seq_lengths, 0, max_seq_width)
            if max_seq_len is not None:
                effective_lengths = np.minimum(seq_lengths, max_seq_len)
            else:
                effective_lengths = seq_lengths.copy()

            all_labels = f[f'{split}_labels'][:]
            max_label_width = int(all_labels.shape[1]) if all_labels.ndim == 2 else 0
            label_lengths = f[f'{split}_label_lengths'][:].astype(np.int32)

            invalid_len = (label_lengths < 0) | (label_lengths > max_label_width)
            if np.any(invalid_len):
                for i in np.flatnonzero(invalid_len):
                    row = all_labels[i]
                    nz = np.flatnonzero(row != 0)
                    label_lengths[i] = int(nz[-1] + 1) if nz.size > 0 else 0
            label_lengths = np.clip(label_lengths, 0, max_label_width)

            valid = (effective_lengths > 0) & (label_lengths > 0)
            valid_idx = np.flatnonzero(valid)

            is_flat = (len(f[f'{split}_sequences'].shape) == 2)
            seqs_ds = f[f'{split}_sequences']

            # ── Chunked preload — dynamic chunk size capped at ~1 GB ──
            row_elems = 1
            for d in seqs_ds.shape[1:]:
                row_elems *= d
            row_bytes = row_elems * 4  # float32
            CHUNK_SIZE = max(1, int(1 * 1024**3 / row_bytes))
            n_total = seqs_ds.shape[0]
            valid_set = set(valid_idx.tolist())
            print(f"[{split}] Row size: {row_bytes/1024**2:.1f} MB, "
                  f"chunk: {CHUNK_SIZE} rows, total: {n_total}", flush=True)

            self._sequences = [None] * len(valid_idx)
            vi_pos = {vi: pos for pos, vi in enumerate(valid_idx)}

            loaded = 0
            for chunk_start in range(0, n_total, CHUNK_SIZE):
                chunk_end = min(chunk_start + CHUNK_SIZE, n_total)
                chunk_valid = [i for i in range(chunk_start, chunk_end) if i in valid_set]
                if not chunk_valid:
                    continue
                chunk_data = seqs_ds[chunk_start:chunk_end]
                for vi in chunk_valid:
                    local = vi - chunk_start
                    sl = int(effective_lengths[vi])
                    if is_flat:
                        seq = chunk_data[local, :sl * self.feature_dim].reshape(sl, self.feature_dim).astype(np.float32)
                    else:
                        seq = chunk_data[local, :sl, :].astype(np.float32)
                    seq = np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0, copy=False)
                    mu = seq.mean(dtype=np.float32)
                    std = seq.std(dtype=np.float32)
                    if not np.isfinite(std) or std < 1e-6:
                        std = 1.0
                    seq = np.clip((seq - mu) / std, -10.0, 10.0)
                    self._sequences[vi_pos[vi]] = seq
                    loaded += 1
                del chunk_data
                print(f"\r  [{split}] {loaded}/{len(valid_idx)} samples loaded", end="", flush=True)
            print()  # newline

            self._labels = [
                all_labels[vi, :int(label_lengths[vi])].astype(np.int32)
                for vi in valid_idx
            ]
            self._seq_lengths = effective_lengths[valid_idx]
            self._label_lengths = label_lengths[valid_idx]
            self._effective_seq_lengths = self._seq_lengths  # alias for BucketBatchSampler

        self.length = len(self._sequences)
        dropped = total_raw - self.length
        if dropped > 0:
            print(f"[{split}] Dropped {dropped}/{total_raw} invalid samples.")
        ram_mb = sum(s.nbytes for s in self._sequences) / 1024**2
        print(f"[{split}] Preloaded {self.length} samples ({ram_mb:.0f} MB RAM)")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self._sequences[idx]),
            torch.from_numpy(self._labels[idx]),
            int(self._seq_lengths[idx]),
            int(self._label_lengths[idx]),
        )


class BucketBatchSampler(Sampler):
    """Shuffle globally, sort locally by length, then batch."""
    def __init__(self, lengths, batch_size, shuffle=True, bucket_multiplier=50):
        self.lengths = np.asarray(lengths, dtype=np.int32)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.bucket_size = max(batch_size, batch_size * bucket_multiplier)

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        indices = np.arange(len(self.lengths))
        if self.shuffle:
            np.random.shuffle(indices)
        batches = []
        for start in range(0, len(indices), self.bucket_size):
            bucket = indices[start:start + self.bucket_size]
            bucket = bucket[np.argsort(self.lengths[bucket], kind='stable')]
            for bs in range(0, len(bucket), self.batch_size):
                batch = bucket[bs:bs + self.batch_size]
                if len(batch) > 0:
                    batches.append(batch.tolist())
        if self.shuffle:
            np.random.shuffle(batches)
        for batch in batches:
            yield batch


def collate_fn(batch, max_seq_len=None):
    sequences, labels, seq_lengths, label_lengths = zip(*batch)
    effective_seq_lengths = []
    trimmed_sequences = []
    for seq, sl in zip(sequences, seq_lengths):
        trimmed_len = min(sl, max_seq_len) if max_seq_len else sl
        trimmed_sequences.append(seq[:trimmed_len])
        effective_seq_lengths.append(trimmed_len)
    max_sl = max(effective_seq_lengths)
    feat_dim = trimmed_sequences[0].size(1)
    bs = len(sequences)
    padded_seqs = torch.zeros(bs, max_sl, feat_dim)
    for i, seq in enumerate(trimmed_sequences):
        padded_seqs[i, :effective_seq_lengths[i]] = seq
    safe_ll = [max(0, min(int(label_lengths[i]), int(labels[i].numel()))) for i in range(bs)]
    max_ll = max(safe_ll) if safe_ll else 0
    padded_labels = torch.zeros(bs, max_ll, dtype=torch.long)
    for i, lbl in enumerate(labels):
        if safe_ll[i] > 0:
            padded_labels[i, :safe_ll[i]] = lbl[:safe_ll[i]]
    return padded_seqs, padded_labels, torch.LongTensor(effective_seq_lengths), torch.LongTensor(safe_ll)


print("Dataset classes defined.")

## 6 — Training & evaluation functions

In [ ]:
def train_epoch(model, loader, optimizer, device, scaler=None,
                grad_accum_steps=1, use_amp=False, pad_idx=0):
    model.train()
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx, label_smoothing=0.1)
    amp_enabled = use_amp
    pbar = tqdm(loader, desc='Training')
    optimizer.zero_grad(set_to_none=True)

    for step, (sequences, labels, seq_lengths, label_lengths) in enumerate(pbar):
        sequences = sequences.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        label_lengths = label_lengths.to(device, non_blocking=True)

        decoder_input, decoder_target, target_lengths = prepare_targets(
            labels, label_lengths,
            sos_idx=model.sos_idx, eos_idx=model.eos_idx, pad_idx=pad_idx,
        )

        with torch.amp.autocast(device_type='cuda', enabled=amp_enabled):
            logits = model(sequences, decoder_input)

        batch_size, max_len, vocab_size = logits.shape
        loss = criterion(logits.reshape(-1, vocab_size), decoder_target.reshape(-1))

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            if amp_enabled:
                amp_enabled = False
                pbar.write('Non-finite AMP loss — falling back to full precision.')
            continue

        loss_bw = loss / grad_accum_steps
        if scaler is not None and amp_enabled:
            scaler.scale(loss_bw).backward()
        else:
            loss_bw.backward()

        should_step = ((step + 1) % grad_accum_steps == 0) or (step + 1 == len(loader))
        if should_step:
            if scaler is not None and amp_enabled:
                scaler.unscale_(optimizer)
                gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not torch.isfinite(gn):
                    optimizer.zero_grad(set_to_none=True)
                    amp_enabled = False
                    pbar.write('Non-finite AMP grads — falling back to full precision.')
                    continue
                scaler.step(optimizer)
                scaler.update()
            else:
                gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not torch.isfinite(gn):
                    optimizer.zero_grad(set_to_none=True)
                    continue
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        predictions = logits.argmax(dim=-1)
        mask = decoder_target != pad_idx
        correct = ((predictions == decoder_target) & mask).sum().item()
        total_correct += correct
        total_tokens += mask.sum().item()
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}',
                         acc=f'{100*correct/max(mask.sum().item(),1):.1f}%')

    avg_loss = total_loss / max(len(loader), 1)
    avg_acc  = 100 * total_correct / total_tokens if total_tokens > 0 else 0
    return avg_loss, avg_acc


def evaluate(model, loader, device, idx_to_gloss, pad_idx=0):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for sequences, labels, seq_lengths, label_lengths in tqdm(loader, desc='Evaluating'):
            sequences = sequences.to(device)
            memory = model.encode(sequences)
            predictions = model.generate(memory, max_length=50)
            for i in range(predictions.size(0)):
                pg = []
                for tok in predictions[i]:
                    t = tok.item()
                    if t == model.eos_idx:
                        break
                    if t >= 3:
                        gi = t - 3
                        if gi in idx_to_gloss:
                            pg.append(idx_to_gloss[gi])
                tg = []
                L = label_lengths[i].item()
                for gi in labels[i, :L]:
                    gi = gi.item()
                    if gi in idx_to_gloss:
                        tg.append(idx_to_gloss[gi])
                all_preds.append(pg)
                all_targets.append(tg)

    exact, tc, tt = 0, 0, 0
    for p, t in zip(all_preds, all_targets):
        if p == t:
            exact += 1
        tc += len(set(p) & set(t))
        tt += len(set(t))
    exact_pct = 100 * exact / max(len(all_preds), 1)
    recall    = 100 * tc / tt if tt > 0 else 0

    print("\nExample predictions:")
    for i in range(min(3, len(all_preds))):
        print(f"  Target:     {' '.join(all_targets[i])}")
        print(f"  Prediction: {' '.join(all_preds[i])}")
    return exact_pct, recall


print("Training functions defined.")

## 7 — Build datasets & model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

train_dataset = H5Dataset(DATA_PATH, 'train', max_seq_len=MAX_SEQ_LEN, h5_cache_mb=H5_CACHE_MB)
val_dataset   = H5Dataset(DATA_PATH, 'val',   max_seq_len=MAX_SEQ_LEN, h5_cache_mb=H5_CACHE_MB)

num_classes    = train_dataset.num_classes
input_features = train_dataset.feature_dim
idx_to_gloss   = train_dataset.idx_to_gloss

print(f"Classes: {num_classes}")
print(f"Input features: {input_features}")
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
print(f"Sample glosses: {[idx_to_gloss[i] for i in range(min(10, num_classes))]}")

In [ ]:
collate = partial(collate_fn, max_seq_len=None)
_pin = device.type == 'cuda'

train_batch_sampler = BucketBatchSampler(
    train_dataset._effective_seq_lengths, batch_size=BATCH_SIZE, shuffle=True,
)
train_loader_kw = dict(
    collate_fn=collate, num_workers=NUM_WORKERS, pin_memory=_pin,
    persistent_workers=(NUM_WORKERS > 0), batch_sampler=train_batch_sampler,
)
if NUM_WORKERS > 0:
    train_loader_kw['prefetch_factor'] = 4
train_loader = DataLoader(train_dataset, **train_loader_kw)

val_workers = max(0, NUM_WORKERS // 2)
val_loader_kw = dict(
    batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate,
    num_workers=val_workers, pin_memory=_pin,
    persistent_workers=(val_workers > 0),
)
if val_workers > 0:
    val_loader_kw['prefetch_factor'] = 4
val_loader = DataLoader(val_dataset, **val_loader_kw)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
model = SignLanguageTransformer(
    num_classes=num_classes,
    input_features=input_features,
    hidden_dim=HIDDEN_DIM,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
    use_channel_attention=USE_CHANNEL_ATTENTION,
    attention_reduction=ATTENTION_REDUCTION,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scaler   = torch.amp.GradScaler('cuda', enabled=(USE_AMP and device.type == 'cuda'))

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Resume from checkpoint ──
best_gloss_recall = 0.0
start_epoch = 1

if RESUME_PATH is not None:
    print(f"\nLoading checkpoint: {RESUME_PATH}")
    ckpt = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_gloss_recall = ckpt.get('best_gloss_recall', 0.0)
    for _ in range(ckpt['epoch']):
        scheduler.step()
    print(f"Resuming from epoch {start_epoch}, best gloss recall so far: {best_gloss_recall:.2f}%\n")
else:
    print("Starting fresh training.")

## 8 — Training loop

In [ ]:
import subprocess

def _upload_checkpoints(epoch_num):
    """Push checkpoints to a private Kaggle dataset for cross-session persistence."""
    meta = {
        "title": CHECKPOINT_DATASET,
        "id": f"{KAGGLE_USERNAME}/{CHECKPOINT_DATASET}",
        "licenses": [{"name": "CC0-1.0"}]
    }
    meta_path = os.path.join(WORKING_CKPT_DIR, "dataset-metadata.json")
    with open(meta_path, "w") as fh:
        json.dump(meta, fh)

    # Try create first; if it already exists, push a new version
    ret = subprocess.run(
        ["kaggle", "datasets", "create", "-p", WORKING_CKPT_DIR, "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    if ret.returncode != 0 and "already exists" in (ret.stdout + ret.stderr).lower():
        ret = subprocess.run(
            ["kaggle", "datasets", "version", "-p", WORKING_CKPT_DIR,
             "-m", f"epoch {epoch_num}", "--dir-mode", "zip"],
            capture_output=True, text=True,
        )
    if ret.returncode == 0:
        print(f"  [upload] Checkpoints pushed to Kaggle dataset (epoch {epoch_num})")
    else:
        print(f"  [upload] WARNING: upload failed — {ret.stderr.strip()[:120]}")


print("=" * 70)
print("ENCODER-DECODER TRANSFORMER TRAINING")
print("=" * 70)
if MAX_SEQ_LEN is not None:
    print(f"Input frame cap: {MAX_SEQ_LEN}")
print(f"Batch: {BATCH_SIZE}, Eval batch: {EVAL_BATCH_SIZE}, "
      f"Grad accum: {GRAD_ACCUM_STEPS}, AMP: {USE_AMP and device.type == 'cuda'}")
print(f"Hidden: {HIDDEN_DIM}, Enc layers: {NUM_ENCODER_LAYERS}, "
      f"Dec layers: {NUM_DECODER_LAYERS}, Heads: {NUM_HEADS}")
print(f"Epochs: {start_epoch} → {EPOCHS}")
if torch.cuda.is_available():
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {mem_gb:.1f} GB")
print("=" * 70)

for epoch in range(start_epoch, EPOCHS + 1):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, device,
        scaler=scaler,
        grad_accum_steps=max(1, GRAD_ACCUM_STEPS),
        use_amp=(USE_AMP and device.type == 'cuda'),
    )
    exact_match, gloss_recall = evaluate(model, val_loader, device, idx_to_gloss)

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Exact Match: {exact_match:.2f}%, Gloss Recall: {gloss_recall:.2f}%")
    print(f"  LR: {scheduler.get_last_lr()[0]:.6f}")
    if torch.cuda.is_available():
        alloc = torch.cuda.max_memory_allocated() / 1024**3
        print(f"  GPU peak mem: {alloc:.2f} GB")
        torch.cuda.reset_peak_memory_stats()
    scheduler.step()

    # ── Save best model ──
    if gloss_recall > best_gloss_recall:
        best_gloss_recall = gloss_recall
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'gloss_recall': gloss_recall,
            'exact_match': exact_match,
            'num_classes': num_classes,
            'input_features': input_features,
            'hidden_dim': HIDDEN_DIM,
            'num_encoder_layers': NUM_ENCODER_LAYERS,
            'num_decoder_layers': NUM_DECODER_LAYERS,
            'use_channel_attention': USE_CHANNEL_ATTENTION,
            'attention_reduction': ATTENTION_REDUCTION,
            'best_gloss_recall': best_gloss_recall,
        }, os.path.join(WORKING_CKPT_DIR, 'best_model.pt'))
        print(f"  >>> Saved best model (gloss_recall={gloss_recall:.2f}%)")

    # ── Save latest checkpoint EVERY epoch (crash-safe resume) ──
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'gloss_recall': gloss_recall,
        'exact_match': exact_match,
        'best_gloss_recall': best_gloss_recall,
        'use_channel_attention': USE_CHANNEL_ATTENTION,
        'attention_reduction': ATTENTION_REDUCTION,
    }, os.path.join(WORKING_CKPT_DIR, 'latest_checkpoint.pt'))
    print(f"  Saved latest_checkpoint.pt (epoch {epoch})")

    # ── Numbered checkpoint every N epochs ──
    if epoch % CKPT_EVERY_N_EPOCHS == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'use_channel_attention': USE_CHANNEL_ATTENTION,
            'attention_reduction': ATTENTION_REDUCTION,
        }, os.path.join(WORKING_CKPT_DIR, f'checkpoint_epoch{epoch}.pt'))

    # ── Auto-upload checkpoints to Kaggle dataset after every epoch ──
    _upload_checkpoints(epoch)

print("\n" + "=" * 70)
print(f"Training complete!  Best gloss recall: {best_gloss_recall:.2f}%")
print("=" * 70)

## 9 — Upload checkpoints to a Kaggle Dataset (cross-session persistence)

Run this cell after training to push your checkpoints to a **private Kaggle
dataset** so that the next session can resume.

In the next session, add the dataset `<your-username>/asl-encdec-checkpoints`
as a notebook input — the resume logic in cell 3 will find it automatically.

In [ ]:
import subprocess, textwrap

# Write dataset-metadata.json required by the Kaggle API
meta = {
    "title": CHECKPOINT_DATASET,
    "id": f"{KAGGLE_USERNAME}/{CHECKPOINT_DATASET}",
    "licenses": [{"name": "CC0-1.0"}]
}
meta_path = os.path.join(WORKING_CKPT_DIR, "dataset-metadata.json")
with open(meta_path, "w") as f:
    json.dump(meta, f)

# Try to create; if it already exists, push a new version instead
ret = subprocess.run(
    ["kaggle", "datasets", "create", "-p", WORKING_CKPT_DIR, "--dir-mode", "zip"],
    capture_output=True, text=True,
)
if ret.returncode != 0 and "already exists" in (ret.stdout + ret.stderr).lower():
    ret = subprocess.run(
        ["kaggle", "datasets", "version", "-p", WORKING_CKPT_DIR,
         "-m", f"epoch {epoch}", "--dir-mode", "zip"],
        capture_output=True, text=True,
    )

print(ret.stdout)
if ret.returncode != 0:
    print("STDERR:", ret.stderr)
else:
    print("Checkpoints uploaded successfully.")

---
## (Optional) Google Drive persistence

If you prefer Google Drive, you can:

1. **Upload** your `best_model.pt` / `latest_checkpoint.pt` to a Google Drive
   folder and make them accessible via a shareable link.
2. Use `gdown` below to **download** them at the start of the next session.
3. After training, **download** the checkpoint files from the Kaggle output
   tab and upload them back to Drive.

In [ ]:
# ── Download checkpoint from Google Drive (optional) ──
#
# 1. Upload your checkpoint file to Google Drive.
# 2. Right-click → Share → "Anyone with the link".
# 3. Copy the file ID from the URL:
#    https://drive.google.com/file/d/<FILE_ID>/view
# 4. Paste it below and uncomment.

# !pip install -q gdown
# import gdown
#
# GDRIVE_CKPT_FILE_ID = "YOUR_GOOGLE_DRIVE_FILE_ID"
# dst = os.path.join(WORKING_CKPT_DIR, "latest_checkpoint.pt")
# if not os.path.exists(dst):
#     gdown.download(id=GDRIVE_CKPT_FILE_ID, output=dst, quiet=False)
#     print(f"Downloaded checkpoint to {dst}")
# else:
#     print(f"Checkpoint already present at {dst}")

In [ ]:
# ── Upload checkpoint to Google Drive (optional) ──
#
# Requires a Google service-account JSON key stored in Kaggle Secrets
# as "GDRIVE_SA_KEY". Skip this if you just download from the Kaggle
# output tab and upload manually.

# from kaggle_secrets import UserSecretsClient
# from pydrive2.auth import GoogleAuth
# from pydrive2.drive import GoogleDrive
# import tempfile, json as _json
#
# secrets = UserSecretsClient()
# sa_json = secrets.get_secret("GDRIVE_SA_KEY")
#
# with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
#     tmp.write(sa_json)
#     sa_path = tmp.name
#
# gauth = GoogleAuth()
# gauth.credentials = GoogleAuth.ServiceAccountCredentials(sa_path, scopes=["https://www.googleapis.com/auth/drive"])
# drive = GoogleDrive(gauth)
#
# GDRIVE_FOLDER_ID = "YOUR_FOLDER_ID"  # Google Drive folder to upload into
# for fname in ["latest_checkpoint.pt", "best_model.pt"]:
#     fpath = os.path.join(WORKING_CKPT_DIR, fname)
#     if os.path.exists(fpath):
#         gfile = drive.CreateFile({"title": fname, "parents": [{"id": GDRIVE_FOLDER_ID}]})
#         gfile.SetContentFile(fpath)
#         gfile.Upload()
#         print(f"Uploaded {fname} to Google Drive.")